In [ ]:
import numpy as np
import pandas as pd

df = data[['Close']].copy()
df.columns = ['Price']

df['Time'] = np.arange(len(df))

df['Lag_1'] = df['Price'].shift(1)

df = df.dropna()

print("Как теперь выглядят наши данные для машинного обучения:")
print(df.head())

In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt

ticker = 'ETH-USD'
data = yf.download(ticker, period='2y')

print("Последние данные с биржи по Эфиру:")
print(data.tail())

plt.figure(figsize=(10, 6))
plt.plot(data.index, data['Close'], label=f'Цена {ticker}', color='purple')
plt.title(f'Исторический график {ticker}')
plt.xlabel('Дата')
plt.ylabel('Цена (USD)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import pandas as pd

df = data[['Close']].copy()
df.columns = ['Price']

df['Time'] = np.arange(len(df))

df['Lag_1'] = df['Price'].shift(1)

df = df.dropna()

print("Готово! Вот так выглядят данные для нашей модели:")
print(df.head())

In [ ]:
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

X = df[['Time', 'Lag_1']]
y = df['Price']

test_size = 30
X_train, X_test = X.iloc[:-test_size], X.iloc[-test_size:]
y_train, y_test = y.iloc[:-test_size], y.iloc[-test_size:]

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

plt.figure(figsize=(12, 6))
plt.plot(y_train.index[-60:], y_train.iloc[-60:], label='Прошлое (Обучение)', color='gray')
plt.plot(y_test.index, y_test, label='Реальная цена', color='purple')
plt.plot(y_test.index, predictions, label='Прогноз модели', color='red', linestyle='dashed', linewidth=2)

plt.title('Прогноз цены Ethereum на экзаменационном периоде (30 дней)')
plt.xlabel('Дата')
plt.ylabel('Цена (USD)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)

xgb_predictions = xgb_model.predict(X_test)

#  Рисуем новый график
plt.figure(figsize=(12, 6))
plt.plot(y_train.index[-60:], y_train.iloc[-60:], label='Прошлое (Обучение)', color='gray')
plt.plot(y_test.index, y_test, label='Реальная цена', color='purple')
plt.plot(y_test.index, xgb_predictions, label='Прогноз XGBoost', color='green', linestyle='dashed', linewidth=2)

plt.title('Прогноз цены Ethereum с помощью XGBoost (30 дней)')
plt.xlabel('Дата')
plt.ylabel('Цена (USD)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
!pip install pandas-datareader

In [ ]:
import pandas_datareader.data as web
import datetime
import pandas as pd
import matplotlib.pyplot as plt

start = datetime.datetime.now() - datetime.timedelta(days=730)
end = datetime.datetime.now()

print("Подключаемся к базе данных ФРС США (FRED)...")

# WALCL - Баланс ФРС (в миллионах $)
# WTREGEN - Счет Казначейства США (в миллионах $)
# RRPONTSYD - Обратное РЕПО (в МИЛЛИАРДАХ $)
fred_data = web.DataReader(['WALCL', 'WTREGEN', 'RRPONTSYD'], 'fred', start, end)

fred_data['RRPONTSYD'] = fred_data['RRPONTSYD'] * 1000

fred_data = fred_data.ffill().dropna()

fred_data['Net_Liquidity'] = fred_data['WALCL'] - fred_data['WTREGEN'] - fred_data['RRPONTSYD']

plt.figure(figsize=(12, 5))
plt.plot(fred_data.index, fred_data['Net_Liquidity'], color='green', label='Чистая Долларовая Ликвидность')
plt.title('WALCL - WTREGEN - RRPONTSYD (Индикатор "Печатного станка")')
plt.xlabel('Дата')
plt.ylabel('Миллионы USD')
plt.legend()
plt.grid(True)
plt.show()

print("\nПоследние значения ликвидности:")
print(fred_data['Net_Liquidity'].tail())

In [ ]:
import yfinance as yf
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

# 1. Скачиваем биржевые котировки (Эфир, S&P 500, 10-летки США)
tickers = ['ETH-USD', '^GSPC', '^TNX']
market_data = yf.download(tickers, period='2y')['Close']
market_data.columns = ['ETH_Price', 'SP500', 'US10Y']

# 2. Объединяем биржу с "печатным станком" ФРС (наша прошлая таблица fred_data)
master_table = market_data.join(fred_data['Net_Liquidity'])

# 3. Заполняем дырки выходных дней (Forward Fill)
master_table = master_table.ffill().dropna()

# 4. FEATURE ENGINEERING: Создаем лаги (взгляд в прошлое)
X = master_table[['ETH_Price', 'SP500', 'US10Y', 'Net_Liquidity']].shift(1)
X.columns = ['Lag_Price', 'Lag_SP500', 'Lag_US10Y', 'Lag_Liquidity']
y = master_table['ETH_Price'] # А цель (y) остается текущей ценой

# Удаляем первую пустую строку из-за сдвига
X = X.dropna()
y = y.loc[X.index]

# 5. Хронологическое разделение (последние 30 дней - на экзамен)
test_size = 30
X_train, X_test = X.iloc[:-test_size], X.iloc[-test_size:]
y_train, y_test = y.iloc[:-test_size], y.iloc[-test_size:]

# 6. Обучаем нашу новую, институциональную модель
smart_xgb = XGBRegressor(n_estimators=1000, learning_rate=0.1, random_state=42)
smart_xgb.fit(X_train, y_train)
smart_predictions = smart_xgb.predict(X_test)

# 7. Рисуем финальный график истины
plt.figure(figsize=(12, 6))
plt.plot(y_train.index[-60:], y_train.iloc[-60:], label='Прошлое (Обучение)', color='gray')
plt.plot(y_test.index, y_test, label='Реальная цена ETH', color='purple')
plt.plot(y_test.index, smart_predictions, label='Прогноз Умного XGBoost', color='green', linewidth=2)

plt.title('Прогноз Эфира (30 дней): XGBoost + Макроэкономика США')
plt.xlabel('Дата')
plt.ylabel('Цена (USD)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 1. Создаем копию нашей базы
df_ta = master_table.copy()

# 2. ДОБАВЛЯЕМ ТЕХНИЧЕСКИЙ АНАЛИЗ 
df_ta['SMA_7'] = df_ta['ETH_Price'].rolling(window=7).mean()
df_ta['SMA_30'] = df_ta['ETH_Price'].rolling(window=30).mean()
df_ta['Volatility_7'] = df_ta['ETH_Price'].rolling(window=7).std()

df_ta = df_ta.dropna()

print("Новые макро-фичи успешно добавлены! 🧠")
print(df_ta[['ETH_Price', 'SMA_7', 'SMA_30', 'Volatility_7']].tail())

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import numpy as np

# 1. СОЗДАЕМ ТОРГОВЫЙ СИГНАЛ (Target)
df_ta['Target'] = (df_ta['ETH_Price'].shift(-1) > df_ta['ETH_Price']).astype(int)

df_ta = df_ta.dropna()

# 2. Выбираем все наши крутые фичи для ИИ
features = ['ETH_Price', 'SP500', 'US10Y', 'Net_Liquidity', 'SMA_7', 'SMA_30', 'Volatility_7']
X = df_ta[features]
y = df_ta['Target']

# 3. Разделение на обучение и экзамен (давай возьмем последние 60 дней для теста)
test_size = 60
X_train, X_test = X.iloc[:-test_size], X.iloc[-test_size:]
y_train, y_test = y.iloc[:-test_size], y.iloc[-test_size:]

# 4. ОБУЧАЕМ ТОРГОВОГО БОТА (XGBClassifier)
clf = XGBClassifier(n_estimators=500, learning_rate=0.05, random_state=42)
clf.fit(X_train, y_train)

signals = clf.predict(X_test)

acc = accuracy_score(y_test, signals)
print(f"Точность предсказаний направления: {acc * 100:.2f}%")


# 5. СЧИТАЕМ ДЕНЬГИ (Бэктест стратегии) 

daily_returns = df_ta['ETH_Price'].pct_change().shift(-1).dropna()
test_returns = daily_returns.iloc[-test_size:]

# Стратегия ИИ: если сигнал 1, мы получаем эту доходность. Если 0 - сидим в кэше (доходность 0)
strategy_returns = signals * test_returns

cumulative_market = (1 + test_returns).cumprod() # Если просто купить и держать
cumulative_strategy = (1 + strategy_returns).cumprod() # Если торговать по сигналам ИИ

plt.figure(figsize=(12, 6))
plt.plot(cumulative_market.index, cumulative_market, label='Стратегия "Купи и держи" (Рынок)', color='purple')
plt.plot(cumulative_strategy.index, cumulative_strategy, label='Торговля по ИИ', color='green', linewidth=2)

plt.title('Бэктест: ИИ против Рынка (за последние 60 дней)')
plt.ylabel('Рост капитала (1.0 = 100% от старта)')
plt.xlabel('Дата')
plt.legend()
plt.grid(True)
plt.show()